# Chaldene Token API — Live Demonstration

This notebook **runs live in JupyterLab** with the Chaldene extension installed. Each section
demonstrates one method of the `IChaldeneService` token API that lets external JupyterLab
extensions interact with visual programming (VP) cells programmatically.

> **How to use**: Run cells top-to-bottom with Shift+Enter.
> VP cells render as interactive node-graph canvases — you can drag nodes around while the API
> demo cells read and write to the same graph.


## Prerequisites

- JupyterLab with Chaldene installed (`pip install chaldene` then restart JupyterLab)
- Run cells **in order** — later cells reference values set by earlier ones

The `%%javascript` cells call the `IChaldeneService` API directly via `window.__chaldene`,
which Chaldene exposes for notebook interaction and browser console debugging.


---
## Step 0 — Verify Chaldene is active

In [ ]:
%%javascript
const svc = window.__chaldene;
if (!svc) {
    element.innerHTML = '<b style="color:red">✗ Chaldene service not found.</b>'
        + '<br>Install the extension and reload JupyterLab, then run this cell again.';
} else {
    const ids = svc.getReadyCellIds();
    element.innerHTML = '<b style="color:green">✓ Chaldene service is ready.</b>'
        + `<br>Currently tracking <b>${ids.length}</b> VP cell(s).`;
    window.__svc = svc;   // cache for subsequent cells
}


---
## Step 1 — Discovery: `isCellReady` and `getReadyCellIds`

Below is a Chaldene VP cell containing a single **read image** node.
When you run it, you should see the interactive node-graph canvas appear.


In [ ]:
{"nodes":[{"id":"0","type":"read_image","position":{"x":100,"y":150},"selected":false,"data":{"specName":"read_image","displayLabel":"read image","description":"Reads a JPEG or PNG image from disk.","inputs":[{"id":"in0","name":"path","type":"string","displayLabel":"file","description":"Path to a JPEG or PNG image.","defaultValue":"sample.png","widget":{"type":"FileInputFromServer","extensions":[".jpg",".jpeg",".png"]}},{"id":"in1","name":"mode","displayLabel":"mode","description":"Colour mode. GRAY produces a single-channel float image.","defaultValue":"GRAY","widget":{"type":"Dropdown","options":["GRAY","RGB"]}}],"outputs":[{"id":"out0","name":"image","type":"image","displayLabel":"image","description":"The loaded image."}]}}],"edges":[]}

Run the API cell below **after** the VP cell above has rendered (the canvas should be visible).


In [ ]:
%%javascript
const svc = window.__svc;

// List every VP cell that is currently live
const ids = svc.getReadyCellIds();
let html = `<b>getReadyCellIds()</b> → ${ids.length} cell(s):<br>`;
ids.forEach((id, i) => {
    const ready = svc.isCellReady(id);
    html += `<code style="display:block;margin:2px 0">[${i}] ${id} — isCellReady: ${ready}</code>`;
});
element.innerHTML = html;

// Store the LAST registered cell ID for use in subsequent steps
// (later cells in the notebook register after earlier ones)
window.__cellId = ids[ids.length - 1];


---
## Step 2 — Reading state: `getGraph`

`getGraph(cellId)` returns a snapshot of the cell's current node graph.
The returned object is a plain `{ nodes, edges }` value — safe to inspect and serialise.


In [ ]:
%%javascript
const svc = window.__svc;
const cellId = window.__cellId;

const graph = svc.getGraph(cellId);
if (!graph) {
    element.textContent = 'getGraph returned undefined — is the cell ready?';
} else {
    let html = `<b>getGraph("${cellId.slice(0,8)}…")</b><br><br>`;
    html += `Nodes: ${graph.nodes.length} &nbsp; Edges: ${graph.edges.length}<br><br>`;
    graph.nodes.forEach(n => {
        const inputs = (n.data.inputs || [])
            .map(h => `${h.id} (${h.name}) = ${JSON.stringify(h.defaultValue ?? '—')}`)
            .join('<br>    ');
        html += `<b>node[${n.id}]</b> type="${n.type}"<br>    ${inputs}<br><br>`;
    });
    element.innerHTML = `<pre style="font-size:11px;line-height:1.4">${html}</pre>`;
}


---
## Step 3 — Replacing a graph: `setGraph`

`setGraph(cellId, graph)` replaces the entire node graph **without unmounting the canvas**.
This is the key benefit over the old remove-and-recreate pattern: no flicker, no lost scroll
position, no lost selection.

After running the cell below, look at the VP cell in Step 1 — a second **auto binarize**
node should appear and connect to the read image node.


In [ ]:
%%javascript
const svc = window.__svc;
const cellId = window.__cellId;

// New graph: read_image → auto binarize
const newGraph = {
    nodes: [
        {
            id: '0', type: 'read_image', position: { x: 100, y: 150 },
            selected: false,
            data: {
                specName: 'read_image', displayLabel: 'read image',
                description: 'Reads a JPEG or PNG image from disk.',
                inputs: [
                    { id: 'in0', name: 'path', type: 'string', displayLabel: 'file',
                      defaultValue: 'sample.png',
                      widget: { type: 'FileInputFromServer',
                                extensions: ['.jpg', '.jpeg', '.png'] } },
                    { id: 'in1', name: 'mode', displayLabel: 'mode', defaultValue: 'GRAY',
                      widget: { type: 'Dropdown', options: ['GRAY', 'RGB'] } }
                ],
                outputs: [{ id: 'out0', name: 'image', type: 'image', displayLabel: 'image' }]
            }
        },
        {
            id: '1', type: 'auto binarize', position: { x: 550, y: 150 },
            selected: false,
            data: {
                specName: 'auto binarize', displayLabel: 'auto binarize',
                description: 'Automatically binarizes using Otsu thresholding.',
                inputs: [{ id: 'in0', name: 'image', type: 'image', displayLabel: 'image' }],
                outputs: [{ id: 'out0', name: 'image', type: 'binary image',
                            displayLabel: 'image' }]
            }
        }
    ],
    edges: [
        { id: '0', source: '0', sourceHandle: 'out0',
          target: '1', targetHandle: 'in0', selected: false }
    ]
};

const ok = svc.setGraph(cellId, newGraph);
element.textContent = ok
    ? '✓ Graph updated — look at the VP cell in Step 1. "auto binarize" has been added.'
    : '✗ setGraph returned false — cell not ready.';


---
## Step 4 — Updating a parameter: `setInputValue`

`setInputValue(cellId, nodeId, handleId, value)` changes a single input handle
without touching the rest of the graph. It is the right tool for parameter sweeps or
any situation where only a value changes and the topology stays the same.

The VP cell below contains a **read image → threshold** pipeline. Run it, then
use the API cell to move the threshold range from `[0.2, 0.8]` to `[0.45, 0.90]`.


In [ ]:
{"nodes":[{"id":"0","type":"read_image","position":{"x":100,"y":150},"selected":false,"data":{"specName":"read_image","displayLabel":"read image","description":"Reads a JPEG or PNG image from disk.","inputs":[{"id":"in0","name":"path","type":"string","displayLabel":"file","description":"Path to a JPEG or PNG image.","defaultValue":"sample.png","widget":{"type":"FileInputFromServer","extensions":[".jpg",".jpeg",".png"]}},{"id":"in1","name":"mode","displayLabel":"mode","description":"Colour mode. GRAY produces a single-channel float image.","defaultValue":"GRAY","widget":{"type":"Dropdown","options":["GRAY","RGB"]}}],"outputs":[{"id":"out0","name":"image","type":"image","displayLabel":"image","description":"The loaded image."}]}},{"id":"1","type":"threshold","position":{"x":550,"y":150},"selected":false,"data":{"specName":"threshold","displayLabel":"threshold","description":"Binarizes by keeping pixels within [lower, upper].","inputs":[{"id":"in0","name":"image","type":"image","displayLabel":"grayscale image","description":"Input image for thresholding."},{"id":"in1","name":"range","type":"tuple2","displayLabel":"Range","description":"Lower and upper threshold bounds.","defaultValue":[0.2,0.8],"widget":{"type":"HistogramRange","min":0,"max":1,"step":0.01}}],"outputs":[{"id":"out0","name":"image","type":"binary image","displayLabel":"binary image","description":"Binary result of thresholding."}]}}],"edges":[{"id":"0","source":"0","sourceHandle":"out0","target":"1","targetHandle":"in0","selected":false}]}

In [ ]:
%%javascript
const svc = window.__svc;

// After running the VP cell above, it becomes the most recently registered cell.
const ids = svc.getReadyCellIds();
window.__threshCellId = ids[ids.length - 1];
const cellId = window.__threshCellId;

// node '1' (threshold), handle 'in1' (range) — change from [0.2, 0.8] to [0.45, 0.90]
const newRange = [0.45, 0.90];
const ok = svc.setInputValue(cellId, '1', 'in1', newRange);
element.textContent = ok
    ? `✓ Threshold range set to [${newRange}]. Look at the Range widget in the VP cell above — it should have moved.`
    : '✗ setInputValue returned false — cell not ready.';


---
## Step 5 — Triggering execution: `run`

`run(cellId)` executes the VP cell — it generates Python code from the node graph
and sends it to the kernel, identical to pressing Shift+Enter on the cell.

Note: `setGraph` and `setInputValue` intentionally do **not** auto-execute. Call
`run()` explicitly when you want results.


In [ ]:
%%javascript
const svc = window.__svc;
const cellId = window.__threshCellId;

// First confirm the graph is what we expect
const graph = svc.getGraph(cellId);
const rangeHandle = graph?.nodes.find(n => n.id === '1')?.data?.inputs?.find(h => h.id === 'in1');
const currentRange = rangeHandle?.defaultValue;

svc.run(cellId);
element.textContent = `✓ Execution triggered for cell ${cellId.slice(0,8)}…`
    + `\nThreshold range at time of run: ${JSON.stringify(currentRange)}`
    + '\n(Check the kernel output below the VP cell — kernel must be running.)';


---
## Step 6 — Observing changes: `graphChanged` signal

`graphChanged` fires every time the user edits the graph interactively in the UI.
Connect a handler to keep external state in sync with the cell.

Run the cell below, then **drag a node in any VP cell** — you should see the signal
fire and display the updated node/edge count.

> The handler disconnects itself after 90 seconds to prevent memory leaks in this demo.


In [ ]:
%%javascript
const svc = window.__svc;
let count = 0;

function onGraphChanged(sender, { cellId, graph }) {
    count++;
    element.innerHTML = [
        `<b>graphChanged fired ${count} time(s)</b>`,
        `<div style="margin:4px 0;font-family:monospace;font-size:11px">`,
        `  cellId : ${cellId}<br>`,
        `  nodes  : ${graph.nodes.length}<br>`,
        `  edges  : ${graph.edges.length}<br>`,
        `  types  : [${graph.nodes.map(n => n.type).join(', ')}]`,
        `</div>`
    ].join('');
}

svc.graphChanged.connect(onGraphChanged);
element.innerHTML = '<i>Listening for graph changes — drag a node in any VP cell above…</i>';

setTimeout(() => {
    svc.graphChanged.disconnect(onGraphChanged);
    element.innerHTML += '<br><i style="color:#888">(listener disconnected after 90 s)</i>';
}, 90000);


---
## Step 7 — Lifecycle signals: `cellReady` and `cellDisposed`

`cellReady` fires when a new VP cell mounts and registers with the service.
`cellDisposed` fires when a VP cell is deleted or the notebook panel closes.

Run the listener below, then **insert a new VP cell** in the notebook — you will see
`cellReady` fire. Delete the cell to see `cellDisposed`.

> **Late-caller pattern**: if your extension activates before VP cells are ready,
> subscribe to `cellReady` and apply your pending state inside the handler.


In [ ]:
%%javascript
const svc = window.__svc;

function onReady(sender, cellId) {
    element.innerHTML += `<div style="color:green">✓ cellReady: <code>${cellId}</code></div>`;
    // Pattern: apply a default graph to every new cell automatically
    // svc.setGraph(cellId, makeDefaultGraph());
}

function onDisposed(sender, cellId) {
    element.innerHTML += `<div style="color:#c00">✗ cellDisposed: <code>${cellId}</code></div>`;
}

svc.cellReady.connect(onReady);
svc.cellDisposed.connect(onDisposed);
element.innerHTML = '<i>Listening for lifecycle events…</i><br>'
    + '<i style="color:#888">Insert or delete a VP cell to see signals fire.</i>';

setTimeout(() => {
    svc.cellReady.disconnect(onReady);
    svc.cellDisposed.disconnect(onDisposed);
    element.innerHTML += '<br><i style="color:#888">(listeners disconnected after 90 s)</i>';
}, 90000);


---
## Bonus — Parameter sweep

The full power of the API shows when you need to run a cell many times with varying
parameters. The cell below sweeps the threshold range across five values, running the
kernel for each one. A real use case: grid-search over processing parameters from an
external optimisation loop or experiment-tracking plugin.

> Requires the kernel to be idle between iterations. This demo inserts a 2-second
> gap between runs; production code should await a proper kernel-idle signal via
> `INotebookTracker`.


In [ ]:
%%javascript
const svc = window.__svc;
const cellId = window.__threshCellId;

const sweepValues = [
    [0.10, 0.50],
    [0.25, 0.65],
    [0.40, 0.80],
    [0.55, 0.90],
    [0.70, 1.00],
];

let step = 0;
element.innerHTML = `<b>Threshold sweep — ${sweepValues.length} steps</b><br>`;

function runStep() {
    if (step >= sweepValues.length) {
        element.innerHTML += '<br><b>✓ Sweep complete.</b>';
        return;
    }
    const range = sweepValues[step];
    svc.setInputValue(cellId, '1', 'in1', range);
    svc.run(cellId);
    element.innerHTML += `<div style="font-size:11px;margin:2px 0">`
        + `step ${step + 1}: range=[${range}] → run()</div>`;
    step++;
    setTimeout(runStep, 2000);   // wait 2 s between runs
}

runStep();


---
## Summary

| Method | What it does | Returns |
|---|---|---|
| `isCellReady(id)` | Checks if VP canvas is mounted | `boolean` |
| `getReadyCellIds()` | Lists all live VP cell IDs | `string[]` |
| `getGraph(id)` | Reads current node graph (snapshot) | `IGraph \| undefined` |
| `setGraph(id, graph)` | Replaces graph without unmounting | `boolean` |
| `setInputValue(id, node, handle, val)` | Updates one input value | `boolean` |
| `run(id)` | Executes the cell in the kernel | `void` |
| `cellReady` signal | Fires on VP canvas mount | `cellId: string` |
| `cellDisposed` signal | Fires on VP canvas teardown | `cellId: string` |
| `graphChanged` signal | Fires on every interactive edit | `{ cellId, graph }` |

The service is available at `window.__chaldene` in this notebook and in the browser console.

For building a full JupyterLab extension that consumes the token via dependency injection,
see `src/tokens.ts` in the Chaldene repository and the TypeScript examples in
`token_api_tutorial.ipynb`.
